# 价格竞猜（The Price is Right）

## 练习目标（第 8 周）

用多 Agent 流水线做「扫特价 → 估真值 → 挑最大折扣 → 通知」：

| 角色 | 职责 |
|------|------|
| Lite Scanner | 返回内置测试交易（不爬网，方便本地跑） |
| Estimator | 经 OpenRouter 估零售价 |
| Planning | 算折扣，超过阈值则通知 |
| Messaging | 控制台日志 + 可选 Pushover 推送 |
| Gradio UI | 表格、日志与演示用向量图 |

## 怎么跑

1. 配置 `.env`：`OPEN_ROUTER_API_KEY`；可选 `PUSHOVER_USER` / `PUSHOVER_TOKEN`
2. 从上到下运行单元格；最后一格启动 Gradio（定时触发扫描）


In [ ]:
# ========== 基础依赖与数据模型（Pydantic）+ Agent 日志基类 ==========

# 标准库：环境变量、JSON 读写、日志
import os
import json
import logging
# 类型标注：列表与可空类型
from typing import List, Optional

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# Pydantic：用 BaseModel 做结构化数据校验；Field 可附带字段说明
from pydantic import BaseModel, Field

# override=True：.env 里的值覆盖已有环境变量
load_dotenv(override=True)


# 单笔交易：描述、挂牌价、链接
class Deal(BaseModel):
    product_description: str
    price: float
    url: str


# 一次扫描选出的多笔交易
class DealSelection(BaseModel):
    deals: List[Deal]


# 机会：在 Deal 上叠加「估值」与「折扣」（估值 - 挂牌价）
class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float


# 所有 Agent 的公共基类：ANSI 颜色 + 带名字的彩色日志
class Agent:
    RED, GREEN, YELLOW, BLUE, MAGENTA, CYAN, WHITE = "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[35m", "\033[36m", "\033[37m"
    BG_BLACK, RESET = "\033[40m", "\033[0m"
    name = ""
    color = "\033[37m"

    # 用黑底 + 代理专属前景色打印，便于在多 Agent 日志里区分来源
    def log(self, message: str):
        logging.info(self.BG_BLACK + self.color + f"[{self.name}] {message}" + self.RESET)


# 配置根日志格式；本模块 logger 供 Framework 使用
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


In [ ]:
# ========== Lite Scanner：用内置 TEST_DEALS 代替真实 RSS 爬取 ==========

# 四笔测试交易（描述 / 价格 / URL 保持原样，供本地联调）
TEST_DEALS = [
    {"product_description": "The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", "price": 178.0, "url": "https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},
    {"product_description": "The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.", "price": 30.0, "url": "https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39"},
    {"product_description": "The Lenovo IdeaPad Slim 5 laptop is powered by a 7th generation AMD Ryzen 5 8645HS 6-core CPU, offering efficient performance for multitasking and demanding applications. It features a 16-inch touch display with a resolution of 1920x1080, ensuring bright and vivid visuals. Accompanied by 16GB of RAM and a 512GB SSD, the laptop provides ample speed and storage for all your files. This model is designed to handle everyday tasks with ease while delivering an enjoyable user experience.", "price": 446.0, "url": "https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-5-7-th-Gen-Ryzen-5-16-Touch-Laptop/485068.html?iref=rss-c39"},
    {"product_description": "The Dell G15 gaming laptop is equipped with a 6th-generation AMD Ryzen 5 7640HS 6-Core CPU, providing powerful performance for gaming and content creation. It features a 15.6-inch 1080p display with a 120Hz refresh rate, allowing for smooth and responsive gameplay. With 16GB of RAM and a substantial 1TB NVMe M.2 SSD, this laptop ensures speedy performance and plenty of storage for games and applications. Additionally, it includes the Nvidia GeForce RTX 3050 GPU for enhanced graphics and gaming experiences.", "price": 650.0, "url": "https://www.dealnews.com/products/Dell/Dell-G15-Ryzen-5-15-6-Gaming-Laptop-w-Nvidia-RTX-3050/485067.html?iref=rss-c39"},
]


# 轻量扫描代理：接口与真实 Scanner 一致，但只返回测试数据
class LiteScannerAgent(Agent):
    name = "Lite Scanner Agent"
    color = Agent.CYAN

    def __init__(self):
        # 就绪日志：标明使用的是内联测试数据
        self.log("Lite Scanner ready (inline test data)")

    # memory 参数保留以兼容 Planning 调用签名（此处未用来过滤）
    def scan(self, memory: List[Opportunity]) -> Optional[DealSelection]:
        self.log("Returning test DealSelection (%d deals)" % len(TEST_DEALS))
        # 字典解包成 Deal，再包进 DealSelection
        return DealSelection(deals=[Deal(**d) for d in TEST_DEALS])


# 供后续 Planning / Framework 注入使用的单例
scanner = LiteScannerAgent()


In [ ]:
# ========== EstimatorAgent：经 OpenRouter 估典型零售价 ==========

# 正则：从模型回复文本里抠出数字价格
import re
# OpenAI 兼容客户端（这里指向 OpenRouter）
from openai import OpenAI


class EstimatorAgent(Agent):

    name = "Estimator Agent"
    color = Agent.YELLOW

    def __init__(self, model: str = "openai/gpt-4o-mini"):
        # 默认走 OpenRouter 上的 gpt-4o-mini 路由名
        self.model = model
        # 从环境变量读 OpenRouter Key；没有则降级为返回 0.0
        key = os.getenv("OPEN_ROUTER_API_KEY")
        if not key:
            self.client = None
            self.log("No OPEN_ROUTER_API_KEY; estimates will return 0.0")
        else:
            # base_url 指向 OpenRouter 的 OpenAI 兼容端点
            self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
        self.log("OpenRouter Estimator ready (model=%s)" % self.model)

    @staticmethod
    def _parse_price(text: str) -> float:
        # 去掉货币符号与千分位逗号，再匹配第一个数字
        text = text.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", text)
        return float(m.group()) if m else 0.0

    def price(self, description: str) -> float:
        # 无客户端时直接返回 0，避免抛错打断流水线
        if not self.client:
            self.log("Skipping estimate (no API key); returning 0.0")
            return 0.0
        # prompt 字符串保持英文原样（影响模型行为，不可翻译）
        prompt = (
            "Estimate the typical retail price in USD for this product. "
            "Reply with only a number (e.g. 299.99), no explanation.\n\n"
            + description.strip()[:2000]
        )
        self.log("Calling OpenRouter for price estimate")
        try:
            # 短回复即可：限制 max_tokens=32
            r = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=32,
            )
            reply = (r.choices[0].message.content or "").strip()
            value = self._parse_price(reply)
            self.log("OpenRouter Estimator completed — $%.2f" % value)
            return value
        except Exception as e:
            # 网络/配额等异常时记日志并返回 0
            self.log("OpenRouter error: %s; returning 0.0" % e)
            return 0.0


# 供 Planning 注入的估价器实例
estimator = EstimatorAgent()


In [ ]:
# ========== Messaging：控制台通知 + 可选 Pushover 推送 ==========

# HTTP 客户端：调用 Pushover REST API
import requests

# Pushover 发送消息的固定端点（URL 不可改）
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"


class ConsoleMessagingAgent(Agent):
    name = "Console Messaging Agent"
    color = Agent.WHITE
    
    def __init__(self):
        # 从环境变量读 Pushover 凭证；缺省为空字符串（仅打日志）
        self.pushover_user = os.getenv("PUSHOVER_USER", "")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "")
        self.log("Console Messaging ready (push=%s)" % bool(self.pushover_token))

    def alert(self, opportunity: Opportunity):
        # 拼一条可读的告警文案：挂牌价 / 估值 / 折扣 + 短描述 + 链接
        msg = (
            f"Deal alert — Price=${opportunity.deal.price:.2f}, "
            f"Estimate=${opportunity.estimate:.2f}, Discount=${opportunity.discount:.2f} | "
            + opportunity.deal.product_description[:80] + "... " + opportunity.deal.url
        )
        # 始终先打到日志，便于无推送环境调试
        self.log("NOTIFY: %s" % msg)
        # 仅当 user 与 token 都齐全时才真正 POST
        if self.pushover_token and self.pushover_user:
            try:
                requests.post(
                    PUSHOVER_URL,
                    data={
                        "user": self.pushover_user,
                        "token": self.pushover_token,
                        "message": msg[:1024],
                        "sound": "cashregister",
                    },
                    timeout=5,
                )
            except Exception as e:
                self.log("Pushover failed: %s" % e)


# 供 Planning 注入的消息代理
messenger = ConsoleMessagingAgent()


In [ ]:
# ========== PlanningAgent：编排「扫描 → 估价 → 挑最优 → 通知」 ==========

class PlanningAgent(Agent):

    name = "Planning Agent"
    color = Agent.GREEN
    # 折扣阈值（美元）：只有超过才告警
    DEAL_THRESHOLD = 50

    def __init__(self, scanner, estimator, messenger):
        self.scanner = scanner
        # 字段名叫 ensemble，这里注入的是 Estimator（接口都是 .price）
        self.ensemble = estimator 
        self.messenger = messenger
        self.log("Planning Agent ready")

    def run(self, deal: Deal) -> Opportunity:
        """Estimate one deal and return an Opportunity."""
        self.log("Pricing one deal")
        # 用产品描述估典型零售价
        estimate = self.ensemble.price(deal.product_description)
        # 折扣 = 估值 - 挂牌价（正数表示「看起来划算」）
        discount = estimate - deal.price
        self.log("Discount $%.2f" % discount)
        return Opportunity(deal=deal, estimate=estimate, discount=discount)

    def plan(self, memory: List[Opportunity]) -> Optional[Opportunity]:
        self.log("Starting plan run")
        # 扫描得到候选交易（本练习为测试数据）
        selection = self.scanner.scan(memory=memory)
        if not selection or not selection.deals:
            self.log("No deals from scanner; done")
            return None
        # 最多估前 5 笔，避免费用/时延爆炸
        opportunities = [self.run(d) for d in selection.deals[:5]]
        # 按折扣从高到低排序，取第一名
        opportunities.sort(key=lambda o: o.discount, reverse=True)
        best = opportunities[0]
        self.log("Best discount $%.2f" % best.discount)
        # 超过阈值才推送通知
        if best.discount > self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        self.log("Plan run complete")
        # 未达阈值则返回 None，表示本轮没有「值得记」的机会
        return best if best.discount > self.DEAL_THRESHOLD else None


# 把前面建好的三个代理串起来
planning_agent = PlanningAgent(scanner, estimator, messenger)


In [ ]:
# ========== DealFramework：持久化 memory + 懒加载 Planner ==========

# 历史机会落盘路径（相对当前工作目录）
PRODUCTS_MEMORY_FILE = "product_memory.json"


class DealFramework:
    def __init__(self, memory_path: str = PRODUCTS_MEMORY_FILE):
        self.memory_path = memory_path
        # 启动时从磁盘读入已发现的 Opportunity 列表
        self.memory = self._read_memory()
        # 延迟初始化：真正 run 时再挂上 planner
        self.planner = None
        logger.info("DealFramework created; memory size = %d", len(self.memory))

    def _read_memory(self) -> List[Opportunity]:
        # 文件存在则反序列化为 Opportunity；否则空列表
        if os.path.exists(self.memory_path):
            with open(self.memory_path, "r") as f:
                data = json.load(f)
            return [Opportunity(**item) for item in data]
        return []

    def _write_memory(self):
        # model_dump：Pydantic v2 转可 JSON 序列化的字典
        data = [o.model_dump() for o in self.memory]
        with open(self.memory_path, "w") as f:
            json.dump(data, f, indent=2)

    def init_agents(self):
        # 只初始化一次，避免重复构造
        if self.planner is None:
            logger.info("Initializing agents")
            # 注意：类名保持原样（ExercisePlanningAgent）
            self.planner = ExercisePlanningAgent(scanner, estimator, messenger)
            logger.info("Agents ready")

    def run(self) -> List[Opportunity]:
        self.init_agents()
        # 跑一轮规划；有结果则追加并落盘
        result = self.planner.plan(memory=self.memory)
        if result:
            self.memory.append(result)
            self._write_memory()
        return self.memory


# 全局框架实例，供下一格与 Gradio 共用
framework = DealFramework()


In [ ]:
# ========== 命令行试跑：执行一轮并打印 memory 摘要 ==========

# 触发扫描→估价→（可能）通知，返回累积的机会列表
opportunities = framework.run()
# 逐条打印：短描述 + 挂牌价 / 估值 / 折扣
for i, opp in enumerate(opportunities):
    print(f"{i+1}. {opp.deal.product_description[:60]}... | Price ${opp.deal.price:.2f} | Est ${opp.estimate:.2f} | Disc ${opp.discount:.2f}")


In [ ]:
# ========== Gradio UI：定时跑 Agent + 日志队列 + 演示用 3D 散点 ==========

# 线程安全队列：主线程读日志，工作线程跑 framework
import queue
import threading
import time
# 随机向量仅用于演示 Plotly 散点（非真实 embedding）
import numpy as np
import gradio as gr
import plotly.graph_objects as go

# ANSI 颜色码 → 后面映射到 HTML span 颜色
BG_BLACK, RED, GREEN, YELLOW, BLUE, MAGENTA, CYAN, WHITE, BG_BLUE, RESET = (
    "\033[40m", "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[35m", "\033[36m", "\033[37m", "\033[44m", "\033[0m"
)
COLOR_MAP = {
    BG_BLACK + RED: "#dd0000", BG_BLACK + GREEN: "#00dd00", BG_BLACK + YELLOW: "#dddd00",
    BG_BLACK + BLUE: "#0000ee", BG_BLACK + MAGENTA: "#aa00dd", BG_BLACK + CYAN: "#00dddd",
    BG_BLACK + WHITE: "#87CEEB", BG_BLUE + WHITE: "#ff7800",
}


def reformat_log(message: str) -> str:
    # 把终端颜色序列替换成 HTML 着色标签
    for k, v in COLOR_MAP.items():
        message = message.replace(k, f'<span style="color: {v}">')
    return message.replace(RESET, "</span>")


class QueueHandler(logging.Handler):
    # 自定义 Handler：emit 时把格式化日志丢进 queue
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def setup_logging(log_queue):
    # 给 root logger 挂上 QueueHandler，供 UI 轮询
    h = QueueHandler(log_queue)
    h.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    root = logging.getLogger()
    root.addHandler(h)
    root.setLevel(logging.INFO)


def html_for(log_data):
    # 只展示最近 20 条，避免 DOM 过大
    out = "<br>".join(log_data[-20:])
    return f"""<div style="height: 420px; overflow-y: auto; border: 1px solid #444; background-color: #1a1a1a; padding: 12px; font-family: monospace; font-size: 13px; color: #fff;">{out}</div>"""


def get_plot():
    # 演示用：随机 3D 点云（不是真实产品向量空间）
    n = 100
    vecs = np.random.randn(n, 3)
    colors = ["red", "blue", "green", "orange"] * (n // 4 + 1)
    fig = go.Figure(data=[go.Scatter3d(x=vecs[:, 0], y=vecs[:, 1], z=vecs[:, 2], mode="markers", marker=dict(size=3, color=colors[:n], opacity=0.7))])
    fig.update_layout(
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="manual", aspectratio=dict(x=2.2, y=2.2, z=1), camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)), bgcolor="#1a1a1a"),
        height=420, margin=dict(r=5, b=5, l=5, t=5), paper_bgcolor="#1a1a1a", font=dict(color="#ffffff"), title="Vector space (demo)",
    )
    return fig


def opportunities_to_table(opps: List[Opportunity]):
    # Opportunity → Gradio Dataframe 行
    if not opps:
        return []
    return [[o.deal.product_description, f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url] for o in opps]


def scan_with_logging(log_data):
    # 后台线程跑 framework.run；前台用 generator yield 刷新日志与表格
    log_queue = queue.Queue()
    result_queue = queue.Queue()
    setup_logging(log_queue)

    def worker():
        framework.run()
        result_queue.put(opportunities_to_table(framework.memory))

    threading.Thread(target=worker).start()
    current = opportunities_to_table(framework.memory)

    while True:
        try:
            # 有新日志就追加并刷新 UI
            msg = log_queue.get_nowait()
            log_data.append(reformat_log(msg))
            current = opportunities_to_table(framework.memory)
            yield log_data, html_for(log_data), current
        except queue.Empty:
            try:
                # 工作线程结束：产出最终表格后 return
                final = result_queue.get_nowait()
                yield log_data, html_for(log_data), final
                return
            except queue.Empty:
                # 两边都空：轻微 sleep，避免忙等
                current = opportunities_to_table(framework.memory)
                yield log_data, html_for(log_data), current
                time.sleep(0.1)


def load_initial_state():
    # 页面加载：空日志 + 当前 memory 表格
    return [], html_for([]), opportunities_to_table(framework.memory)


def handle_selection(selected_index: gr.SelectData):
    # 点击表格行 → 对应该行 Opportunity 再发一次 alert
    row = selected_index.index[0]
    if row < len(framework.memory):
        framework.planner.messenger.alert(framework.memory[row])
        return f"Alert sent for: {framework.memory[row].deal.product_description[:60]}..."
    return "Invalid selection"


# 定时器间隔（秒）：仅由 Timer 触发扫描，无手动按钮
TIMER_SECONDS = 60

# 启动 UI 前先确保 planner 已挂好（供行点击用）
framework.init_agents()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    # State：跨回调持久保存日志列表
    log_data = gr.State([])
    gr.Markdown(
        "<div style='text-align: center; font-size: 28px; font-weight: bold; margin: 20px 0;'>The Price is Right - Week 8 Exercise</div>"
        + "<div style='text-align: center; font-size: 16px; color: #666; margin-bottom: 20px;'>Table above, logs + plot below. Agent runs on timer only (every " + str(TIMER_SECONDS) + " s).</div>"
    )
    with gr.Row():
        opportunities_table = gr.Dataframe(
            headers=["Product Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True, column_widths=[6, 1, 1, 1, 3], row_count=10, col_count=5, max_height=420, interactive=False,
        )
    with gr.Row():
        with gr.Column(scale=1):
            logs_display = gr.HTML(label="Agent Logs")
        with gr.Column(scale=1):
            vector_plot = gr.Plot(value=get_plot(), show_label=False)
    # 首次加载填充表格
    ui.load(load_initial_state, inputs=[], outputs=[log_data, logs_display, opportunities_table])
    # 周期性 tick → 跑 scan_with_logging
    timer = gr.Timer(value=TIMER_SECONDS, active=True)
    timer.tick(scan_with_logging, inputs=[log_data], outputs=[log_data, logs_display, opportunities_table])
    selection_feedback = gr.Textbox(visible=False)
    opportunities_table.select(handle_selection, inputs=[], outputs=[selection_feedback])
# share=False：不生成公网链接；inbrowser=True：尝试自动打开浏览器
ui.launch(share=False, inbrowser=True)
